# Full Chain Training + Inference (Unified Pipeline)

This notebook runs the complete 12-stage chain in one pipeline:
group-classifier -> group-splitter -> endpoint-regression -> event-splitter -> pion-stop -> positron-angle,
with training and inference for each model in sequence.

## 1) Environment Setup

In [1]:
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from pioneerml.common.integration.zenml import load_step_output
from pioneerml.common.integration.zenml import utils as zenml_utils
from pioneerml.pipelines.full_chain import full_chain_pipeline

PROJECT_ROOT = Path(zenml_utils.find_project_root())
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)

MODEL_CHAIN = (
    "group_classifier",
    "group_splitter",
    "endpoint_regression",
    "event_splitter",
    "pion_stop",
    "positron_angle",
)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## 2) Select Input Data and Output Workspace

By default this notebook builds an aligned 8-row shard for quick iteration.
Set `USE_SMALL_SUBSET = False` to run against the full source parquet.

In [2]:
def _select_aligned_row_indices(
    *,
    main_file: Path,
    count: int = 8,
    min_hits: int = 12,
    min_positron_hits: int = 2,
) -> list[int]:
    table = pq.read_table(main_file, columns=["hits_pdg_id", "hits_particle_mask"])
    pdg_lists = table["hits_pdg_id"].to_pylist() if "hits_pdg_id" in table.column_names else [None] * table.num_rows
    mask_lists = table["hits_particle_mask"].to_pylist()

    lengths = np.asarray([len(v) if v is not None else 0 for v in mask_lists], dtype=np.int64)
    positron_counts = np.asarray(
        [int(sum(int(x) == -11 for x in (row or []))) for row in pdg_lists],
        dtype=np.int64,
    )

    eligible = np.flatnonzero((lengths >= int(min_hits)) & (positron_counts >= int(min_positron_hits)))
    if eligible.size == 0:
        raise RuntimeError(
            f"No rows with >= {min_hits} hits and >= {min_positron_hits} positron hits found in {main_file}."
        )
    order = np.argsort(lengths[eligible], kind="stable")
    chosen = eligible[order][: int(count)]
    if chosen.size == 0:
        raise RuntimeError(f"Failed to select aligned rows from {main_file}.")
    return [int(v) for v in chosen.tolist()]


def _write_aligned_subset(*, src: Path, dst: Path, row_indices: list[int]) -> None:
    table = pq.read_table(src)
    index_arr = pa.array([int(v) for v in row_indices], type=pa.int64())
    pq.write_table(table.take(index_arr), dst)


USE_SMALL_SUBSET = False
SMALL_ROW_COUNT = 8

src_main = PROJECT_ROOT / "data" / "ml_output_000.parquet"
workspace_dir = PROJECT_ROOT / "artifacts" / "full_chain_notebook"
models_dir = workspace_dir / "models"
preds_dir = workspace_dir / "preds"
workspace_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
preds_dir.mkdir(parents=True, exist_ok=True)

if USE_SMALL_SUBSET:
    main_path = workspace_dir / "ml_output_000_small.parquet"
    selected_rows = _select_aligned_row_indices(main_file=src_main, count=int(SMALL_ROW_COUNT))
    _write_aligned_subset(src=src_main, dst=main_path, row_indices=selected_rows)
else:
    main_path = src_main

main_sources = [str(main_path)]

pred_paths = {
    "group_classifier": preds_dir / "group_classifier_preds.parquet",
    "group_splitter": preds_dir / "group_splitter_preds.parquet",
    "endpoint_regression": preds_dir / "endpoint_regressor_preds.parquet",
    "event_splitter": preds_dir / "event_splitter_preds.parquet",
    "pion_stop": preds_dir / "pion_stop_preds.parquet",
    "positron_angle": preds_dir / "positron_angle_preds.parquet",
}

for p in pred_paths.values():
    if p.exists():
        p.unlink()

print("main_path:", main_path)
print("workspace_dir:", workspace_dir)


main_path: /workspace/data/ml_output_000.parquet
workspace_dir: /workspace/artifacts/full_chain_notebook


## 3) Reusable Config Helpers

In [3]:
def make_trainer_cfg(*, max_epochs: int = 1, show_progress: bool = True) -> dict[str, Any]:
    return {
        "type": "lightning_module",
        "config": {
            "trainer_kwargs": {
                "enable_progress_bar": bool(show_progress),
                "max_epochs": int(max_epochs),
                "gradient_clip_val": 2.0,
            },
            "early_stopping": {
                "enabled": True,
                "type": "relative",
                "config": {
                    "monitor": "val_loss",
                    "mode": "min",
                    "patience": 3,
                    "min_delta": 0.0,
                    "strict": True,
                    "check_finite": True,
                    "verbose": False,
                },
            },
        },
    }


def make_loader_defaults(
    *,
    loader_type: str,
    mode: str,
    batch_size: int | None,
    sample_fraction: float = 1.0,
    chunk_row_groups: int = 2,
    chunk_workers: int = 0,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    cfg: dict[str, Any] = {
        "batch_size": batch_size,
        "mode": str(mode),
        "chunk_row_groups": int(chunk_row_groups),
        "chunk_workers": int(chunk_workers),
        "edge_template_cache_enabled": False,
        "edge_template_cache_max_entries": None,
        "sample_fraction": float(sample_fraction),
        "train_fraction": 0.75,
        "val_fraction": 0.125,
        "test_fraction": 0.125,
        "split_seed": 0,
    }
    if extra:
        cfg.update(dict(extra))
    return {"type": str(loader_type), "config": cfg}


def make_loader_manager_cfg(*, input_sources_spec: dict[str, Any], defaults: dict[str, Any], loaders: dict[str, Any]) -> dict[str, Any]:
    return {
        "config": {
            "input_sources_spec": dict(input_sources_spec),
            "input_backend": {"type": "parquet", "config": {}},
            "defaults": dict(defaults),
            "loaders": dict(loaders),
        }
    }


def make_training_stage_cfg(
    *,
    architecture_cfg: dict[str, Any],
    module_cfg: dict[str, Any],
    loader_type: str,
    input_sources_spec: dict[str, Any],
    loader_extra: dict[str, Any] | None,
    evaluator_cfg: dict[str, Any],
    export_dir: Path,
    export_prefix: str,
    hpo_storage: str,
    hpo_study_name: str,
    search_space: dict[str, Any],
    train_batch_size: int | None = None,
    max_epochs: int = 1,
    hpo_trials: int = 1,
    sample_fraction: float = 0.05,
    evaluate_enabled: bool = False,
) -> dict[str, Any]:
    hpo_defaults = make_loader_defaults(
        loader_type=loader_type,
        mode="train",
        batch_size=None,
        sample_fraction=float(sample_fraction),
        chunk_row_groups=2,
        chunk_workers=0,
        extra=loader_extra,
    )
    train_defaults = make_loader_defaults(
        loader_type=loader_type,
        mode="train",
        batch_size=(None if train_batch_size is None else max(1, int(train_batch_size))),
        sample_fraction=float(sample_fraction),
        chunk_row_groups=2,
        chunk_workers=0,
        extra=loader_extra,
    )
    export_defaults = make_loader_defaults(
        loader_type=loader_type,
        mode="train",
        batch_size=1,
        sample_fraction=1.0,
        chunk_row_groups=1,
        chunk_workers=0,
        extra=loader_extra,
    )

    return {
        "hpo": {
            "architecture": architecture_cfg,
            "module": module_cfg,
            "trainer": make_trainer_cfg(max_epochs=int(max_epochs), show_progress=True),
            "loader_manager": make_loader_manager_cfg(
                input_sources_spec=input_sources_spec,
                defaults=hpo_defaults,
                loaders={
                    "train_loader": {"config": {"mode": "train", "split": "train", "shuffle_batches": True}},
                    "val_loader": {"config": {"mode": "train", "split": "val", "shuffle_batches": False}},
                },
            ),
            "hpo": {
                "type": "config",
                "config": {
                    "enabled": True,
                    "n_trials": int(hpo_trials),
                    "direction": "minimize",
                    "seed": 0,
                    "study_name": str(hpo_study_name),
                    "storage": str(hpo_storage),
                    "fallback_dir": None,
                    "allow_schema_fallback": True,
                    "objective": {"type": "val_epoch", "config": {}},
                    "search_space": {
                        "type": "config",
                        "config": {"search_space": dict(search_space)},
                    },
                },
            },
        },
        "train": {
            "architecture": architecture_cfg,
            "module": module_cfg,
            "trainer": make_trainer_cfg(max_epochs=int(max_epochs), show_progress=True),
            "loader_manager": make_loader_manager_cfg(
                input_sources_spec=input_sources_spec,
                defaults=train_defaults,
                loaders={
                    "train_loader": {"config": {"mode": "train", "split": "train", "shuffle_batches": True}},
                    "val_loader": {"config": {"mode": "train", "split": "val", "shuffle_batches": False}},
                },
            ),
        },
        "evaluate": {
            "enabled": bool(evaluate_enabled),
            "evaluator": dict(evaluator_cfg),
            "loader_manager": make_loader_manager_cfg(
                input_sources_spec=input_sources_spec,
                defaults=train_defaults,
                loaders={"test_loader": {"config": {"mode": "train", "split": "test", "shuffle_batches": False}}},
            ),
        },
        "export": {
            "exporter": {
                "type": "torchscript",
                "config": {
                    "enabled": True,
                    "export_dir": str(export_dir),
                    "filename_prefix": str(export_prefix),
                    "prefer_cuda": True,
                },
            },
            "loader_manager": make_loader_manager_cfg(
                input_sources_spec=input_sources_spec,
                defaults=export_defaults,
                loaders={"export_loader": {"config": {"mode": "train", "shuffle_batches": False}}},
            ),
        },
    }


def make_inference_stage_cfg(
    *,
    model_subdir: str,
    writer_type: str,
    loader_type: str,
    input_sources_spec: dict[str, Any],
    loader_extra: dict[str, Any] | None,
    output_dir: Path,
    output_path: Path,
    inference_batch_size: int = 8,
) -> dict[str, Any]:
    inference_defaults = make_loader_defaults(
        loader_type=loader_type,
        mode="inference",
        batch_size=int(inference_batch_size),
        sample_fraction=1.0,
        chunk_row_groups=2,
        chunk_workers=0,
        extra=loader_extra,
    )
    inference_defaults["config"]["mode"] = "inference"

    return {
        "model_handle_builder": {
            "model_handle": {
                "type": "script",
                "config": {
                    "model_subdir": str(model_subdir),
                    "model_path": None,
                },
            }
        },
        "inference": {
            "runtime": {"prefer_cuda": True},
            "writer": {
                "type": str(writer_type),
                "config": {
                    "output_backend": {"type": "parquet", "config": {"target_row_group_rows": 1024}},
                    "fallback_output_dir": str(output_dir),
                    "output_dir": str(output_dir),
                    "output_path": str(output_path),
                    "streaming": True,
                    "write_timestamped": False,
                    "timestamp": None,
                    "writer_params": {},
                },
            },
            "loader_manager": {
                "type": "config",
                "config": {
                    "input_sources_spec": dict(input_sources_spec),
                    "input_backend": {"type": "parquet", "config": {}},
                    "defaults": inference_defaults,
                    "loaders": {
                        "inference_loader": {
                            "config": {
                                "mode": "inference",
                                "shuffle_batches": False,
                                "log_diagnostics": False,
                            }
                        }
                    },
                },
            },
        },
    }


## 4) Build Full Chain Config

In [4]:
HPO_TRIALS = 10
MAX_EPOCHS = 10
TRAIN_BATCH_SIZE = None
INFERENCE_BATCH_SIZE = 32
SAMPLE_FRACTION = 0.05
EVALUATE_ENABLED = True

input_gc = {"main_sources": list(main_sources), "optional_sources_by_name": {}, "source_type": "file"}
input_gs = {"main_sources": list(main_sources), "optional_sources_by_name": {"group_probs": [str(pred_paths["group_classifier"])]}, "source_type": "file"}
input_ep = {
    "main_sources": list(main_sources),
    "optional_sources_by_name": {"group_probs": [str(pred_paths["group_classifier"])], "group_splitter": [str(pred_paths["group_splitter"])]},
    "source_type": "file",
}
input_ev = {
    "main_sources": list(main_sources),
    "optional_sources_by_name": {
        "group_probs": [str(pred_paths["group_classifier"])],
        "group_splitter": [str(pred_paths["group_splitter"])],
        "endpoint": [str(pred_paths["endpoint_regression"])],
    },
    "source_type": "file",
}
input_ps = {
    "main_sources": list(main_sources),
    "optional_sources_by_name": {
        "group_probs": [str(pred_paths["group_classifier"])],
        "group_splitter": [str(pred_paths["group_splitter"])],
        "endpoint": [str(pred_paths["endpoint_regression"])],
        "event_splitter": [str(pred_paths["event_splitter"])],
    },
    "source_type": "file",
}
input_pa = {
    "main_sources": list(main_sources),
    "optional_sources_by_name": {
        "group_probs": [str(pred_paths["group_classifier"])],
        "group_splitter": [str(pred_paths["group_splitter"])],
        "endpoint": [str(pred_paths["endpoint_regression"])],
        "event_splitter": [str(pred_paths["event_splitter"])],
        "pion_stop": [str(pred_paths["pion_stop"])],
    },
    "source_type": "file",
}

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
optuna_dir = PROJECT_ROOT / ".optuna"
optuna_dir.mkdir(parents=True, exist_ok=True)

def make_study(model_key: str) -> tuple[str, str]:
    study_name = f"full_chain_notebook_{model_key}_{stamp}"
    return study_name, f"sqlite:///{optuna_dir / f'{study_name}.db'}"

pipeline_config: dict[str, Any] = {}

specs = {
    "group_classifier": {
        "architecture": {"type": "group_classifier", "config": {"node_dim": 4, "edge_dim": 4, "graph_dim": 0, "hidden": 128, "heads": 4, "num_blocks": 2, "dropout": 0.1}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "bce_with_logits", "config": {}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "group_classifier",
        "writer_type": "group_classification",
        "model_subdir": "groupclassifier",
        "input": input_gc,
        "loader_extra": None,
        "evaluator": {"type": "simple_classification", "config": {"threshold": 0.5, "metrics": ["binary_classification_from_tensors"], "plots": []}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 1, "max_exp": 3}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
    "group_splitter": {
        "architecture": {"type": "group_splitter", "config": {"node_dim": 4, "edge_dim": 4, "graph_dim": 3, "hidden": 128, "heads": 4, "layers": 3, "dropout": 0.1, "num_classes": 3}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "bce_with_logits", "config": {}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "group_splitter",
        "writer_type": "group_splitting",
        "model_subdir": "groupsplitter",
        "input": input_gs,
        "loader_extra": None,
        "evaluator": {"type": "simple_classification", "config": {"threshold": 0.5, "metrics": ["binary_classification_from_tensors"], "plots": []}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 1, "max_exp": 3}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
    "endpoint_regression": {
        "architecture": {"type": "endpoint_regressor", "config": {"node_dim": 7, "graph_dim": 3, "splitter_prob_dimension": 0, "edge_dim": 4, "hidden": 192, "heads": 4, "layers": 3, "dropout": 0.1, "output_dim": 18}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "mse", "config": {}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "endpoint_regression",
        "writer_type": "endpoint_regression",
        "model_subdir": "endpoint_regressor",
        "input": input_ep,
        "loader_extra": None,
        "evaluator": {"type": "simple_regression", "config": {"plot_dir": str(workspace_dir / "plots" / "endpoint_regression")}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 1, "max_exp": 3}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
    "event_splitter": {
        "architecture": {"type": "event_splitter", "config": {"node_dim": 4, "group_prob_dimension": 3, "splitter_prob_dimension": 3, "endpoint_dimension": 18, "edge_attr_dimension": 5, "hidden": 192, "heads": 4, "layers": 3, "dropout": 0.1}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "bce_with_logits", "config": {}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "event_splitter",
        "writer_type": "event_splitter",
        "model_subdir": "event_splitter",
        "input": input_ev,
        "loader_extra": {"use_group_probs": True, "use_splitter_probs": True, "use_endpoint_preds": True},
        "evaluator": {"type": "simple_classification", "config": {"threshold": 0.5, "metrics": ["binary_classification_from_counters"], "plots": []}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 0, "max_exp": 2}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
    "pion_stop": {
        "architecture": {"type": "pion_stop_regressor", "config": {"node_dim": 4, "graph_dim": 3, "splitter_prob_dimension": 3, "endpoint_pred_dimension": 18, "event_affinity_dimension": 3, "pion_stop_pred_dimension": 0, "edge_dim": 4, "hidden": 192, "heads": 4, "layers": 3, "dropout": 0.1, "output_dim": 9}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "quantile_pinball", "config": {}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "pion_stop",
        "writer_type": "pion_stop",
        "model_subdir": "pion_stop_regression",
        "input": input_ps,
        "loader_extra": {"use_group_probs": True, "use_splitter_probs": True, "use_endpoint_preds": True, "use_event_splitter_affinity": True, "use_pion_stop_preds": False},
        "evaluator": {"type": "simple_regression", "config": {"plot_dir": str(workspace_dir / "plots" / "pion_stop")}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 1, "max_exp": 3}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
    "positron_angle": {
        "architecture": {"type": "positron_angle_regressor", "config": {"node_dim": 4, "graph_dim": 3, "splitter_prob_dimension": 3, "endpoint_pred_dimension": 18, "event_affinity_dimension": 3, "pion_stop_pred_dimension": 3, "edge_dim": 4, "hidden": 192, "heads": 4, "layers": 3, "dropout": 0.1, "output_dim": 9}},
        "module": {"type": "graph_lightning", "config": {"loss": {"type": "quantile_angular", "config": {"num_outputs": 3, "quantiles": (0.16, 0.50, 0.84), "pinball_weight": 0.0, "angular_weight": 1.0, "unit_norm_weight": 0.0, "normalize_target": False, "clamp_dot": False}}, "lr": 1e-3, "weight_decay": 1e-4, "scheduler_step_size": 2, "scheduler_gamma": 0.5}},
        "loader_type": "positron_angle",
        "writer_type": "positron_angle",
        "model_subdir": "positron_angle_regression",
        "input": input_pa,
        "loader_extra": {"use_group_probs": True, "use_splitter_probs": True, "use_endpoint_preds": True, "use_event_splitter_affinity": True, "use_pion_stop_preds": True, "training_relevant_only": True, "min_relevant_hits": 2},
        "evaluator": {"type": "simple_regression", "config": {"plot_dir": str(workspace_dir / "plots" / "positron_angle")}},
        "search_space": {"batch_size": {"type": "exponent_int", "base": 2, "min_exp": 1, "max_exp": 3}, "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True}, "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-3, "log": True}},
    },
}

for model_key, spec in specs.items():
    study_name, storage = make_study(model_key)
    pipeline_config[model_key] = {
        "training": make_training_stage_cfg(
            architecture_cfg=spec["architecture"],
            module_cfg=spec["module"],
            loader_type=spec["loader_type"],
            input_sources_spec=spec["input"],
            loader_extra=spec["loader_extra"],
            evaluator_cfg=spec["evaluator"],
            export_dir=models_dir / model_key,
            export_prefix=f"{model_key}_chain",
            hpo_storage=storage,
            hpo_study_name=study_name,
            search_space=spec["search_space"],
            train_batch_size=TRAIN_BATCH_SIZE,
            max_epochs=int(MAX_EPOCHS),
            hpo_trials=int(HPO_TRIALS),
            sample_fraction=float(SAMPLE_FRACTION),
            evaluate_enabled=bool(EVALUATE_ENABLED),
        ),
        "inference": make_inference_stage_cfg(
            model_subdir=spec["model_subdir"],
            writer_type=spec["writer_type"],
            loader_type=spec["loader_type"],
            input_sources_spec=spec["input"],
            loader_extra=spec["loader_extra"],
            output_dir=preds_dir / model_key,
            output_path=pred_paths[model_key],
            inference_batch_size=int(INFERENCE_BATCH_SIZE),
        ),
    }

print("Configured models:", list(pipeline_config.keys()))


Configured models: ['group_classifier', 'group_splitter', 'endpoint_regression', 'event_splitter', 'pion_stop', 'positron_angle']


## 5) Run Full Chain Pipeline

In [5]:
run = full_chain_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


Initiating a new run for the pipeline: full_chain_pipeline.
Caching is disabled by default for full_chain_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step run_training_stage_step has started.


[I 2026-03-25 19:04:47,260] A new study created in RDB with name: full_chain_notebook_group_classifier_20260325_190445


[run_training_stage_step] GPU available: True (cuda), used: True
[run_training_stage_step] TPU available: False, using: 0 TPU cores
[run_training_stage_step] You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set torch.set_float32_matmul_precision('medium' | 'high') which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
[run_training_stage_step] LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ GroupClassifierStereo │  630 K │ train │     0 │
│ 1 │ loss_fn │ BCEWithLogitsLoss     │      0 │ train │     0 │
└───┴─────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 630 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 630 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 57                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ PositronAngleModel  │  1.5 M │ train │     0 │
│ 1 │ loss_fn │ QuantileAngularLoss │      0 │ train │     0 │
└───┴─────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.5 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 62                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

[run_training_stage_step_6] Trainer.fit stopped: max_epochs=10 reached.


[run_training_stage_step_6] No materializer is registered for type <class 'pioneerml.common.integration.pytorch.modules.graph_lightning_module.GraphLightningModule'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.common.integration.pytorch.modules.graph_lightning_module.GraphLightningModule'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step run_training_stage_step_6 has finished in 1m52s.
Step run_cleanup_step_11 has started.
Step run_cleanup_step_11 has finished in 1.207s.
Step run_inference_stage_step_6 has started.
[run_inference_stage_step_6] No materializer is registered for type <class 'pioneerml.common.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'>, so the default Pickle mate

## 6) Inspect Stage Outputs

In [6]:
def _step_name(base: str, idx1: int) -> str:
    return base if idx1 == 1 else f"{base}_{idx1}"

print("run_id:", run.id)

for idx, model_key in enumerate(MODEL_CHAIN, start=1):
    train_step = _step_name("run_training_stage_step", idx)
    infer_step = _step_name("run_inference_stage_step", idx)

    train_out = load_step_output(run, train_step)
    infer_out = load_step_output(run, infer_step)

    export_payload = (dict(train_out or {}).get("export") if isinstance(train_out, dict) else None) or {}
    inference_payload = (dict(infer_out or {}).get("inference") if isinstance(infer_out, dict) else None) or {}

    pred_paths_out = list(dict(inference_payload or {}).get("predictions_paths") or [])
    if not pred_paths_out and isinstance(inference_payload, dict) and inference_payload.get("predictions_path"):
        pred_paths_out = [str(inference_payload["predictions_path"])]

    print(f"[{model_key}] export:", export_payload)
    print(f"[{model_key}] predictions:", pred_paths_out)

final_pred_path = pred_paths["positron_angle"]
if final_pred_path.exists():
    tbl = pq.read_table(final_pred_path)
    print("final_positron_rows:", tbl.num_rows)
    print("final_positron_columns:", tbl.column_names)
else:
    print("Final positron-angle prediction file not found yet:", final_pred_path)


run_id: a1418624-d493-41dc-a174-7e09d3ccf6c5
[group_classifier] export: {'torchscript_path': '/workspace/artifacts/full_chain_notebook/models/group_classifier/group_classifier_chain_20260325_190706_torchscript.pt', 'metadata_path': '/workspace/artifacts/full_chain_notebook/models/group_classifier/group_classifier_chain_20260325_190706_meta.json', 'export_type': 'script', 'exporter_type': 'torchscript'}
[group_classifier] predictions: ['/workspace/artifacts/full_chain_notebook/preds/group_classifier_preds.parquet']
[group_splitter] export: {'torchscript_path': '/workspace/artifacts/full_chain_notebook/models/group_splitter/group_splitter_chain_20260325_190855_torchscript.pt', 'metadata_path': '/workspace/artifacts/full_chain_notebook/models/group_splitter/group_splitter_chain_20260325_190855_meta.json', 'export_type': 'script', 'exporter_type': 'torchscript'}
[group_splitter] predictions: ['/workspace/artifacts/full_chain_notebook/preds/group_splitter_preds.parquet']
[endpoint_regressio